In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

In [2]:
train_df = pd.read_csv("/kaggle/input/datasets/checkdizzcode/training-xlm/train_data.csv")
test_df = pd.read_csv("/kaggle/input/datasets/checkdizzcode/training-xlm/test_data.csv")

train_df['sarcasm_type'] = train_df['sarcasm_type'].fillna("None")
test_df['sarcasm_type'] = test_df['sarcasm_type'].fillna("None")

# Chỉ Undersample (Cắt bớt Non-Sarcastic) cho bằng với tổng số Sarcastic.
# TUYỆT ĐỐI KHÔNG Oversampling để tránh model bị học vẹt data rác.
df_sar = train_df[train_df['sarcasm_label'] == 'Sarcastic']
total_sar = len(df_sar)

df_non = train_df[train_df['sarcasm_label'] == 'Non-Sarcastic']
df_non_sampled = df_non.sample(n=total_sar, random_state=42) # Tỷ lệ 1:1 chuẩn xác

# Gộp lại
train_df = pd.concat([df_non_sampled, df_sar]).sample(frac=1.0, random_state=42).reset_index(drop=True)
# Tạo 6 nhãn gộp
def get_6_labels(row):
    if row['sarcasm_label'] == 'Non-Sarcastic': return 'Non-Sarcastic'
    if pd.isna(row['sarcasm_type']) or row['sarcasm_type'] == 'None': return 'Propositional Contradiction'
    return row['sarcasm_type']

train_df['final_label'] = train_df.apply(get_6_labels, axis=1)
test_df['final_label'] = test_df.apply(get_6_labels, axis=1)

# Chuyển nhãn thành số
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['final_label'])
test_df['label'] = le.transform(test_df['final_label'])


train_ds = Dataset.from_pandas(train_df[['video_core_content', 'comment', 'label']])
test_ds = Dataset.from_pandas(test_df[['video_core_content', 'comment', 'label']])

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truyền tách biệt video và comment để Tokenizer hiểu 2 vế
    return tokenizer(
        examples["video_core_content"], 
        examples["comment"], 
        padding="max_length", 
        truncation="only_first", 
        max_length=512
    )

train_tokenized = train_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1046 [00:00<?, ? examples/s]

Map:   0%|          | 0/796 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:



# 2. ĐỊNH NGHĨA LẠI TRAINER
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")3.1
        
        # Ép kiểu Float để chống nổ số trên FP16
        ce_loss = F.cross_entropy(logits.view(-1, self.model.config.num_labels).float(), labels.view(-1), reduction='none')
        
        # Tính pt (xác suất dự đoán trúng)
        pt = torch.exp(-ce_loss)
        
        # Công thức Focal Loss (gamma = 2.0 giúp dồn lực học vào các câu mỉa mai khó)
        gamma = 2.0
        focal_loss = ((1 - pt) ** gamma * ce_loss).mean()
        
        return (focal_loss, outputs) if return_outputs else focal_loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, predictions, average='macro')}
# 3. SET LEARNING RATE THẤP XUỐNG
training_args = TrainingArguments(
    output_dir="./xlm_results",
    learning_rate=3e-5,          # LR thấp giúp Loss tuy cao nhưng ổn định, không bị nổ
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=5,            # Cho học lâu hơn 1 chút
    weight_decay=0.01,
    eval_strategy="epoch", 
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True, 
    report_to="none"
)

trainer = FocalLossTrainer(
    model=model, 
    args=training_args, 
    train_dataset=train_tokenized, 
    eval_dataset=test_tokenized, 
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Dừng nếu 3 vòng không cải thiện F1
)

print("⏳ Bắt đầu quá trình Train với Trọng số...")
trainer.train()

⏳ Bắt đầu quá trình Train với Trọng số...


Epoch,Training Loss,Validation Loss,F1 Macro
1,No log,0.443666,0.167136
2,No log,0.475220,0.185499
3,No log,0.329339,0.194981
4,No log,0.316128,0.222067
5,No log,0.340185,0.220256


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=165, training_loss=0.7993692109079072, metrics={'train_runtime': 407.9142, 'train_samples_per_second': 12.821, 'train_steps_per_second': 0.404, 'total_flos': 1376120240271360.0, 'train_loss': 0.7993692109079072, 'epoch': 5.0})

In [4]:


print("📊 Đang xuất báo cáo và lưu file...")
predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

target_names = le.inverse_transform([0, 1, 2, 3, 4, 5])

print("="*60)
print("🏆 BÁO CÁO KẾT QUẢ XML")
print("="*60)
print(classification_report(y_true, y_pred, target_names=target_names))

# Lấy lại tên nhãn text từ số
predicted_merged = le.inverse_transform(y_pred)

# Tách ngược lại thành Label và Type để chuẩn format
test_df['predicted_label'] = ['Non-Sarcastic' if p == 'Non-Sarcastic' else 'Sarcastic' for p in predicted_merged]
test_df['predicted_type'] = ['None' if p == 'Non-Sarcastic' else p for p in predicted_merged]

output_file = "test_result_XML_Roberta.csv"
test_df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu kết quả chi tiết của PhoBERT vào: {output_file}")

📊 Đang xuất báo cáo và lưu file...


🏆 BÁO CÁO KẾT QUẢ XML
                             precision    recall  f1-score   support

               Hypothetical       0.12      0.67      0.20        12
      Lexical Contradiction       0.08      0.11      0.09        19
              Non-Sarcastic       0.97      0.85      0.91       738
Propositional Contradiction       0.09      0.25      0.13        20
        Rhetorical Question       0.00      0.00      0.00         5
                Wh-Question       0.00      0.00      0.00         2

                   accuracy                           0.81       796
                  macro avg       0.21      0.31      0.22       796
               weighted avg       0.91      0.81      0.85       796

✅ Đã lưu kết quả chi tiết của PhoBERT vào: test_result_XML_Roberta.csv


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
